In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib, os, numpy as np, pandas as pd
from lightgbm import LGBMRegressor
from scipy.stats import rankdata, pearsonr


# ── Shared utilities (same as Step 1) ─────────────────────────────────────────

def cs_rank(s):
    n = len(s)
    if n < 2: return s * 0.0
    r = s.rank(method="average")
    return (r - 1) / (n - 1) - 0.5

def cs_fill(df, cols):
    df = df.copy()
    df[cols] = df.groupby("moon")[cols].transform(lambda s: s.fillna(s.median()))
    df[cols] = df[cols].fillna(0.0)
    return df

def quick_ic(X, y_series, feature_cols, n_moons=60):
    """Fast cross-sectional IC for a sample of moons — used for feature ranking."""
    moons = X["moon"].values
    unique = np.unique(moons)
    if len(unique) > n_moons:
        unique = np.random.default_rng(42).choice(unique, n_moons, replace=False)
    ic_sum = np.zeros(len(feature_cols))
    ic_cnt = np.zeros(len(feature_cols))
    for m in unique:
        mask = moons == m
        yt = y_series.values[mask]
        if yt.std() < 1e-10: continue
        Xm = X.iloc[mask][feature_cols].values
        for j in range(len(feature_cols)):
            xf = Xm[:, j]
            if xf.std() < 1e-10: continue
            r = np.corrcoef(xf, yt)[0, 1]
            if not np.isnan(r):
                ic_sum[j] += r
                ic_cnt[j] += 1
    safe = np.where(ic_cnt > 0, ic_cnt, 1)
    mean_ic = ic_sum / safe
    mean_ic[ic_cnt == 0] = 0.0
    order = np.argsort(np.abs(mean_ic))[::-1]
    return [feature_cols[i] for i in order], np.abs(mean_ic)[order]

def make_train_infer(feature_cols_to_use):
    """Returns train/infer closures using the specified feature set."""

    def train(X_train, y_train, model_directory_path, loop_moon, embargo):
        import joblib, os, numpy as np, pandas as pd
        from lightgbm import LGBMRegressor

        feats = [c for c in feature_cols_to_use if c in X_train.columns]
        df = X_train.merge(y_train[["moon","id","target"]], on=["moon","id"])
        df = df.dropna(subset=["target"])
        df[feats] = df.groupby("moon")[feats].transform(lambda s: s.fillna(s.median()))
        df[feats] = df[feats].fillna(0.0)
        df["target_rank"] = df.groupby("moon")["target"].transform(
            lambda s: (s.rank(method="average") - 1) / max(len(s)-1,1) - 0.5)
        model = LGBMRegressor(n_estimators=200, learning_rate=0.05,
                               random_state=42, n_jobs=-1, verbose=-1)
        model.fit(df[feats], df["target_rank"])
        joblib.dump({"model": model, "features": feats},
                    os.path.join(model_directory_path, "model.joblib"))

    def infer(X_test, model_directory_path, loop_moon, embargo):
        import joblib, os, numpy as np, pandas as pd
        from scipy.stats import rankdata
        obj   = joblib.load(os.path.join(model_directory_path, "model.joblib"))
        feats = obj["features"]
        X = X_test.copy()
        for c in feats:
            med = X[c].median()
            X[c] = X[c].fillna(med if not np.isnan(med) else 0.0)
        raw = obj["model"].predict(X[feats])
        n = len(raw)
        final = (rankdata(raw) - 1) / max(n-1,1) - 0.5
        return pd.DataFrame({"moon": X_test["moon"].values,
                              "id": X_test["id"].values, "prediction": final})
    return train, infer

def evaluate(train_fn, infer_fn, X_tr, y_tr, X_te, y_te, splits, label):
    import os
    model_dir = f"./model_ablation_{label}"
    os.makedirs(model_dir, exist_ok=True)
    train_fn(X_tr, y_tr, model_dir, loop_moon=0, embargo=4)
    results = []
    for moon in splits["reduced_cloud"]:
        pred = infer_fn(X_te[X_te["moon"]==moon], model_dir, loop_moon=moon, embargo=4)
        results.append(pred)
    preds = pd.concat(results)
    corrs = []
    for moon in splits["reduced_cloud"]:
        merged = preds[preds["moon"]==moon].merge(
            y_te[y_te["moon"]==moon], on=["moon","id"])
        if len(merged) > 1:
            r, _ = pearsonr(merged["prediction"], merged["target"])
            corrs.append(r)
    mean_ic = float(np.mean(corrs)) if corrs else 0.0
    print(f"  [{label:20s}]  Mean IC = {mean_ic:.4f}  per-moon: {[round(c,3) for c in corrs]}")
    return mean_ic


In [ ]:
import time

all_feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]
print(f"Total features: {len(all_feature_cols)}")

# Rank features by cross-sectional IC on a sample of training moons
print("Ranking features by IC (sample of 60 moons) ...")
t0 = time.perf_counter()
df_merge = X_train.merge(y_train[["moon","id","target"]], on=["moon","id"]).dropna(subset=["target"])
ranked_feats, ranked_ics = quick_ic(df_merge, df_merge["target"], all_feature_cols, n_moons=60)
print(f"  Done in {time.perf_counter()-t0:.1f}s")
print(f"  Top 10 by |IC|: {ranked_feats[:10]}")
print(f"  IC values:      {[round(v,4) for v in ranked_ics[:10]]}")

# Define feature subsets to test
subsets = {
    "all_1150":  all_feature_cols,
    "top_100":   ranked_feats[:100],
    "top_50":    ranked_feats[:50],
    "top_34_ic": ranked_feats[:34],   # our offline analysis sweet spot
    "top_20":    ranked_feats[:20],
    "first_100": [f"Feature_{i}" for i in range(1, 101)],  # original baseline
}

print("\nRunning ablation study ...")
print("-" * 60)
results_ablation = {}
for label, feats in subsets.items():
    train_fn, infer_fn = make_train_infer(feats)
    ic = evaluate(train_fn, infer_fn, X_train, y_train,
                  X_test_cloud, y_test_cloud, splits, label)
    results_ablation[label] = ic

print("\n" + "=" * 60)
print("ABLATION SUMMARY")
print("=" * 60)
for label, ic in sorted(results_ablation.items(), key=lambda x: -x[1]):
    bar = "█" * int(max(ic, 0) * 200)
    print(f"  {label:20s}: {ic:+.4f}  {bar}")
print("\nBaseline (all feats, no rank-norm): ~0.0200")
